# Train_v4

**Change from v3**:
- Using more data as trainset : split ts_index from 2880 to 3500

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))


In [ ]:
import lightgbm as lgb
import pandas as pd
import numpy as np
import gc
from src.score import weighted_rmse_score as score


In [3]:
def set_cat(df, cat_features):
    for feat in cat_features:
        df[feat] = df[feat].astype('category')
    return df


## Config

In [ ]:
HORIZONS = [1, 3, 10, 25]

# Codes whose predictions are forced to 0 (both val and test)
ZERO_CODES = ['MLAAMU3K', 'EP12UF2K', '1HEMHZK2', 'SJZP0OVU', '83EG83KQ']

VERSION ="v4"

params = {
    'objective': 'regression',
    'boosting_type': 'gbdt',
    'metric': 'rmse',
    'num_leaves': 80,
    'min_child_samples':200,
    'lambda_l2':10,
    'learning_rate': 0.01,
    'bagging_fraction': 0.7,   # pick 70% of data to train each tree
    'bagging_freq': 5,          # perform bagging every 5 iterations
    'feature_fraction': 0.8,   # pick 80% of features to train each tree
    'seed': 42,
    'verbosity': -1
}


## Load & Prepare Data

In [ ]:
df = pd.read_parquet(PROJECT_ROOT / "data" / "train.parquet")
cat_features = ['code', 'sub_category', 'horizon']
features = [feat for feat in df.columns if feat not in ['id', 'sub_code', 'ts_index', 'weight', 'y_target']]
df = set_cat(df, cat_features)


## Time-based Train/Val Split
Split is computed on the full dataset so each horizon uses the same boundary.

In [6]:
split_index = 3500
train_df = df[df['ts_index'] <= split_index]
val_df   = df[df['ts_index'] >  split_index]
print(f"Train size: {len(train_df):,} | Val size: {len(val_df):,} | Split ts_index: {split_index}")
del df
gc.collect()


Train size: 5,172,152 | Val size: 165,262 | Split ts_index: 3500


20

## Train One Model per Horizon

In [ ]:
models      = {}   # horizon -> trained booster
importances = {}   # horizon -> importance DataFrame

for h in HORIZONS:
    print(f"\n{'='*60}")
    print(f"  Training horizon = {h}")
    print(f"{'='*60}")

    # --- Subset by horizon ---
    h_train = train_df[train_df['horizon'] == h].copy()
    h_val   = val_df[val_df['horizon'] == h].copy()
    print(f"  Train rows: {len(h_train):,} | Val rows: {len(h_val):,}")

    # --- LightGBM Datasets ---
    # horizon is constant within each subset, but keep it as a feature
    # for consistency with v1; LGB will simply ignore it (zero split gain)
    dtrain = lgb.Dataset(
        h_train[features],
        label=h_train['y_target'],
        weight=h_train['weight'],
        categorical_feature=cat_features,
        free_raw_data=False
    )
    dval = lgb.Dataset(
        h_val[features],
        label=h_val['y_target'],
        weight=h_val['weight'],
        categorical_feature=cat_features,
        reference=dtrain,
        free_raw_data=False
    )

    # --- Train ---
    model = lgb.train(
        params,
        dtrain,
        valid_sets=[dtrain, dval],
        valid_names=['train', 'val'],
        num_boost_round=4200,
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=100)
        ]
    )
    models[h] = model

    # --- Save model ---
    model_path = PROJECT_ROOT / "models" / f"lgb_model_{VERSION}_horizon{h}.txt"
    model.save_model(model_path, num_iteration=model.best_iteration)
    print(f"  Model saved -> {model_path}  (best iteration: {model.best_iteration})")

    # --- Feature importance ---
    imp = pd.DataFrame({
        'feature': features,
        'importance': model.feature_importance(importance_type='gain')
    }).sort_values('importance', ascending=False)
    importances[h] = imp
    print(f"\n  Top-10 features (horizon={h}):")
    print(imp.head(10).to_string(index=False))



  Training horizon = 1
  Train rows: 1,351,193 | Val rows: 43,460
Training until validation scores don't improve for 200 rounds
[100]	train's rmse: 0.000976334	val's rmse: 0.0011362
[200]	train's rmse: 0.000970759	val's rmse: 0.00113567
[300]	train's rmse: 0.000966602	val's rmse: 0.00113536
[400]	train's rmse: 0.0009631	val's rmse: 0.00113424
[500]	train's rmse: 0.000960093	val's rmse: 0.0011341
[600]	train's rmse: 0.000957189	val's rmse: 0.00113391
[700]	train's rmse: 0.00095437	val's rmse: 0.0011339
[800]	train's rmse: 0.000951953	val's rmse: 0.00113393
[900]	train's rmse: 0.000949885	val's rmse: 0.0011338
[1000]	train's rmse: 0.000947817	val's rmse: 0.00113354
[1100]	train's rmse: 0.00094576	val's rmse: 0.00113352
[1200]	train's rmse: 0.000943698	val's rmse: 0.00113359
Early stopping, best iteration is:
[1037]	train's rmse: 0.000946938	val's rmse: 0.00113338
  Model saved -> /home/lingod/kaggle_projects/ts_forecasting/models/lgb_model_v4_horizon1.txt  (best iteration: 1037)

  Top-

## Train Scoring (per-horizon + overall)
Zero-code override is applied before scoring.aa

In [8]:
# Collect all train predictions into a single aligned series
train_preds_series = pd.Series(index=train_df.index, dtype=float)

horizon_scores = {}

for h in HORIZONS:
    h_train  = train_df[train_df['horizon'] == h].copy()
    model  = models[h]

    preds  = model.predict(h_train[features], num_iteration=model.best_iteration)
    preds  = pd.Series(preds, index=h_train.index)

    # Apply zero-code override on val
    zero_mask = h_train['code'].isin(ZERO_CODES)
    preds[zero_mask] = 0.0

    train_preds_series[h_train.index] = preds

    h_score = score(h_train['y_target'], preds, h_train['weight'])
    horizon_scores[h] = h_score
    print(f"  Horizon {h:>2d} | Val Score: {h_score:.4f}  "
          f"(zero-coded rows: {zero_mask.sum():,})")

# Overall val score (all horizons together)
overall_val_score = score(train_df['y_target'], train_preds_series, train_df['weight'])
print(f"\n  Overall Val Score: {overall_val_score:.4f}")


  Horizon  1 | Val Score: 0.2430  (zero-coded rows: 266,358)
  Horizon  3 | Val Score: 0.2336  (zero-coded rows: 265,104)
  Horizon 10 | Val Score: 0.3926  (zero-coded rows: 255,715)
  Horizon 25 | Val Score: 0.4482  (zero-coded rows: 230,583)

  Overall Val Score: 0.4017


## Validation Scoring (per-horizon + overall)
Zero-code override is applied before scoring.

In [9]:
# Collect all val predictions into a single aligned series
val_preds_series = pd.Series(index=val_df.index, dtype=float)

horizon_scores = {}

for h in HORIZONS:
    h_val  = val_df[val_df['horizon'] == h].copy()
    model  = models[h]

    preds  = model.predict(h_val[features], num_iteration=model.best_iteration)
    preds  = pd.Series(preds, index=h_val.index)

    # Apply zero-code override on val
    zero_mask = h_val['code'].isin(ZERO_CODES)
    preds[zero_mask] = 0.0

    val_preds_series[h_val.index] = preds

    h_score = score(h_val['y_target'], preds, h_val['weight'])
    horizon_scores[h] = h_score
    print(f"  Horizon {h:>2d} | Val Score: {h_score:.4f}  "
          f"(zero-coded rows: {zero_mask.sum():,})")

# Overall val score (all horizons together)
overall_val_score = score(val_df['y_target'], val_preds_series, val_df['weight'])
print(f"\n  Overall Val Score: {overall_val_score:.4f}")


  Horizon  1 | Val Score: 0.0642  (zero-coded rows: 7,675)
  Horizon  3 | Val Score: 0.0966  (zero-coded rows: 7,618)
  Horizon 10 | Val Score: 0.2014  (zero-coded rows: 7,238)
  Horizon 25 | Val Score: 0.2217  (zero-coded rows: 6,609)

  Overall Val Score: 0.1962


In [10]:
del train_df, val_df, train_preds_series, val_preds_series
gc.collect()


32

## Test Predictions
Each test row is predicted by the model matching its horizon.
Zero-code override is applied after prediction.

In [ ]:
df_test = pd.read_parquet(PROJECT_ROOT / "data" / "test.parquet")
df_test = set_cat(df_test, cat_features)


In [12]:
test_preds_series = pd.Series(index=df_test.index, dtype=float)

for h in HORIZONS:
    h_test    = df_test[df_test['horizon'] == h]
    model     = models[h]

    preds     = model.predict(h_test[features], num_iteration=model.best_iteration)
    preds     = pd.Series(preds, index=h_test.index)

    # Apply zero-code override on test
    zero_mask = h_test['code'].isin(ZERO_CODES)
    preds[zero_mask] = 0.0

    test_preds_series[h_test.index] = preds
    print(f"  Horizon {h:>2d} | Test rows: {len(h_test):,}  "
          f"(zero-coded rows: {zero_mask.sum():,})")

# Sanity check: no NaNs
assert test_preds_series.isna().sum() == 0, "Some test rows were not predicted!"
print(f"\n  Total test predictions: {len(test_preds_series):,}")


  Horizon  1 | Test rows: 379,617  (zero-coded rows: 69,005)
  Horizon  3 | Test rows: 376,558  (zero-coded rows: 68,523)
  Horizon 10 | Test rows: 362,057  (zero-coded rows: 65,675)
  Horizon 25 | Test rows: 328,875  (zero-coded rows: 59,033)

  Total test predictions: 1,447,107


## Build & Save Submission

In [13]:
prediction_df = pd.DataFrame({
    'id':         df_test['id'].values,
    'prediction': test_preds_series.values
})

prediction_df.info(max_cols=10)
prediction_df.head()


<class 'pandas.DataFrame'>
RangeIndex: 1447107 entries, 0 to 1447106
Data columns (total 2 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   id          1447107 non-null  str    
 1   prediction  1447107 non-null  float64
dtypes: float64(1), str(1)
memory usage: 73.8 MB


,id,prediction
0,W2MW3G2L__495MGHFJ__PZ9S1Z4V__3__3647,-0.048202
1,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3647,-0.077029
2,W2MW3G2L__495MGHFJ__PZ9S1Z4V__25__3647,-0.482904
3,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647,-0.006606
4,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3648,-0.092299


In [ ]:
prediction_df.to_csv(
    PROJECT_ROOT / "submissions" / f"submission_{VERSION}.csv",
    index=False
)
print("Submission saved.")


Submission saved.
